# Ruta Transversal B — Análisis de comportamiento con t-SNE

Proyecto **Riopaila Castilla** (TAMML — Tarea 1). Analiza el historial de interacciones del
agente de **OpenFang** para descubrir clústeres de **intenciones de usuario**.

**Pipeline:** sesiones JSONL de OpenFang → embeddings (OpenAI `text-embedding-3-small`) →
t-SNE (2D) → KMeans → visualización e interpretación.

> Requiere: `numpy scikit-learn matplotlib openai` y haber sembrado interacciones con
> `python src/scripts/seed_interactions.py` (o tener historial real de usuarios).


In [ ]:
import sys, json
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

# Reutiliza las funciones del script de análisis
sys.path.insert(0, str(Path.cwd().parent / 'src' / 'scripts'))
import tsne_analysis as ta
print('Sesiones JSONL encontradas:', len(ta.SESSIONS_GLOB))

## 1. Extracción del historial de interacciones (JSONL de OpenFang)

In [ ]:
msgs = ta.extract_user_messages()
labels_map = json.loads(ta.LABELS_PATH.read_text(encoding='utf-8')) if ta.LABELS_PATH.exists() else {}
intents = [labels_map.get(m, 'otro') for m in msgs]
print(f'Mensajes de usuario únicos: {len(msgs)}')
for m in msgs[:5]:
    print(' -', m)

## 2. Vectorización (embeddings de OpenAI)

In [ ]:
X = ta.embed(msgs, ta.load_env_key())
print('Matriz de embeddings:', X.shape)  # (n_mensajes, 1536)

## 3. Reducción de dimensionalidad con t-SNE

In [ ]:
n = len(msgs)
perp = max(5, min(30, n // 3))
coords = TSNE(n_components=2, perplexity=perp, init='pca',
              learning_rate='auto', random_state=7).fit_transform(X)
print('Proyección 2D:', coords.shape, '| perplexity =', perp)

## 4. Clústeres (KMeans) y visualización

In [ ]:
k = max(2, len(set(i for i in intents if i != 'otro')) or 8)
clusters = KMeans(n_clusters=k, n_init=10, random_state=7).fit_predict(coords)

uniq = sorted(set(intents))
cmap = plt.get_cmap('tab10')
cbi = {it: cmap(i % 10) for i, it in enumerate(uniq)}

fig, ax = plt.subplots(1, 2, figsize=(16, 7))
for it in uniq:
    p = coords[[j for j, x in enumerate(intents) if x == it]]
    ax[0].scatter(p[:, 0], p[:, 1], s=60, alpha=.8, color=cbi[it], label=it)
ax[0].set_title('t-SNE por intención real'); ax[0].legend(fontsize=8)
ax[1].scatter(coords[:, 0], coords[:, 1], s=60, alpha=.8, c=clusters, cmap='tab10')
ax[1].set_title(f't-SNE + KMeans (k={k})')
fig.suptitle('Riopaila Castilla — Análisis de intenciones de usuario (OpenFang)')
plt.tight_layout(); plt.show()

## 5. Interpretación de los clústeres

In [ ]:
for c in range(k):
    idx = [j for j in range(n) if clusters[j] == c]
    counts = Counter(intents[j] for j in idx)
    dom, dom_n = counts.most_common(1)[0]
    print(f"Clúster {c}: {len(idx)} msgs | dominante = '{dom}' (pureza {dom_n/len(idx):.0%})")
    for e in [msgs[j] for j in idx[:3]]:
        print('     -', e)